In [1]:
import torch
from torch.utils.data import DataLoader, Dataset
import pydicom as dicom
import pandas as pd
import numpy as np
import utils
import unet

import skimage
import scipy

import matplotlib.pyplot as plt

In [2]:
device = torch.device("xpu" if torch.xpu.is_available() else "cpu")

# path to the root folder of the data
ROOT_FILEPATH = "/run/media/gianluca/EXTERNAL_US/CBIS-DDSM"

In [3]:
class CBIS_Dataset(Dataset):
    def __init__(self, dataset_filepath, train, transform=None):
        self.transform = transform
        lesions_df = pd.read_csv(dataset_filepath)
        
        # keeping only MLO and masses
        lesions_df = lesions_df[
            (lesions_df["image view"] == "MLO") & 
            (lesions_df["kind"] == "Mass")
        ]
        
        # based on parameter, keep only training or test instances
        if train:
            lesions_df = lesions_df[lesions_df["training or test"] == "training"]
        else:
            lesions_df = lesions_df[lesions_df["training or test"] == "Test"]

        self.lesions_df = lesions_df.head(10)
        
        self.images = []
        self.masks = []
        
        # load and preprocess the lesions
        for index, row in self.lesions_df.iterrows():
            image_tensor = torch.load(
                f"{ROOT_FILEPATH}/" + row["preprocessed fullimage tensor filepath"],
                weights_only=True
            ).float()
            mask_tensor = torch.load(
                f"{ROOT_FILEPATH}/" + row["preprocessed mask tensor filepath"],
                weights_only=True
            ).float()

            self.images.append(image_tensor)
            self.masks.append(mask_tensor)
            
        self.n_elements = len(self.images)
    
    def __len__(self):
        return self.n_elements

    def __getitem__(self, idx):
        image, mask = self.images[idx], self.masks[idx]

        if self.transform:
            image = self.transform(image)

        return image, mask

In [4]:
learning_rate = 1e-3
batch_size = 2
epochs = 3

# loading data
train_data = CBIS_Dataset(dataset_filepath=f"{ROOT_FILEPATH}/lesions.csv", train=True)
test_data = CBIS_Dataset(dataset_filepath=f"{ROOT_FILEPATH}/lesions.csv", train=False)

train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

# defyning loss function
loss_fn = unet.dice_loss

model = unet.UNet(n_class=2)
#model.to(torch.float16) # Converti tutti i parametri del modello (inclusi i bias) in float16
model.to(device)

# defyning optimizer
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=learning_rate
)

In [8]:
# training loop
for t in range(epochs):
    print(f"Epoch {t+1}\n------------")
    unet.train_loop(train_dataloader, model, loss_fn, optimizer, batch_size, device)
    unet.test_loop(test_dataloader, model, loss_fn, device)

Epoch 1
------------
onednn_verbose,v1,info,oneDNN v3.9.1 (commit 80a3a8e745d2f0186e674b0af9332fd6e074c94f)
onednn_verbose,v1,info,cpu,runtime:threadpool,nthr:7
onednn_verbose,v1,info,cpu,isa:Intel AVX2 with Intel DL Boost
onednn_verbose,v1,info,gpu,runtime:DPC++
onednn_verbose,v1,info,gpu,engine,sycl gpu device count:1 
onednn_verbose,v1,info,gpu,engine,0,backend:Level Zero,name:Intel(R) Graphics [0x7d41],driver_version:1.3.30049,binary_kernels:enabled
onednn_verbose,v1,info,graph,backend,0:dnnl_backend
onednn_verbose,v1,primitive,info,template:operation,engine,primitive,implementation,prop_kind,memory_descriptors,attributes,auxiliary,problem_desc,exec_time
onednn_verbose,v1,graph,info,template:operation,engine,partition_id,partition_kind,op_names,data_formats,logical_tensors,fpmath_mode,implementation,backend,exec_time
onednn_verbose,v1,common,error,runtime,Intel(R) Graphics [0x7d41] OpenCL device not found,src/gpu/intel/sycl/utils.cpp:136


RuntimeError: could not create a primitive

In [14]:
X = torch.rand(1, 1, 572, 572).to(device)
y = torch.rand(1, 2, 388, 388).to(device)

In [15]:
model.train()
with torch.no_grad():
    pred = model(X)

In [16]:
loss = loss_fn(pred, y)